# Vocal Tract Demo — OLA FIR (`pink_trombone_ola`)

Same examples as `tract_demo.ipynb`, but uses the OLA FIR approximation instead of
the sequential waveguide.  Each cell compares both methods:

- **Reference** (`pink_trombone`): exact sequential Kelly-Lochbaum waveguide
- **OLA FIR** (`pink_trombone_ola`): one impulse response per control frame, overlap-add convolution

See the bottom of this notebook for a speed benchmark.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time

import torch
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from samuel.pink_trombone import (
    pink_trombone,
    pink_trombone_ola,
    PARAM_NAMES,
    N_PARAMS,
    SAMPLE_RATE,
    SAMPLES_PER_FRAME,
    _compute_diameter_profile,
    _TRACT_N,
)

In [ ]:
CONTROL_RATE = SAMPLE_RATE / SAMPLES_PER_FRAME


def make_params(duration_s=2.0, **overrides):
    T = int(duration_s * CONTROL_RATE)
    defaults = {
        "noise": 0.1,
        "frequency": 350,
        "tenseness": 0.6,
        "intensity": 1,
        "loudness": 1,
        "tongueIndex": 12.9,
        "tongueDiameter": 2.43,
        "vibratoWobble": 0,
        "vibratoFrequency": 6,
        "vibratoGain": 0.0,
        "tractLength": 44,
        "constrictionIndex": 44.0,
        "constrictionDiameter": 3.0,
    }
    params = torch.zeros(1, T, N_PARAMS)
    for i, name in enumerate(PARAM_NAMES):
        val = overrides.get(name, defaults.get(name, 0))
        if isinstance(val, torch.Tensor):
            params[0, :, i] = val[:T]
        else:
            params[0, :, i] = val
    return params


def play_both(params, seed=0, ir_length=4096, normalize=True, label=None):
    """Synthesize with both methods, print timing, and play both."""
    with torch.no_grad():
        t0 = time.perf_counter()
        ref = pink_trombone(params, seed=seed)
        t_ref = time.perf_counter() - t0

        t0 = time.perf_counter()
        ola = pink_trombone_ola(params, seed=seed, ir_length=ir_length)
        t_ola = time.perf_counter() - t0

    if label:
        print(f"── {label} ──")

    ref_rms = ref.pow(2).mean().sqrt().item()
    err_rms = (ref - ola).pow(2).mean().sqrt().item()
    rel_err = err_rms / (ref_rms + 1e-9)
    print(f"  reference: {t_ref*1000:.0f} ms  |  OLA FIR: {t_ola*1000:.0f} ms  |  relative error: {rel_err:.3f}")

    def norm(w):
        return w / w.abs().max().clamp(min=1e-6) * 0.9 if normalize else w

    print("  Reference:")
    display(Audio(norm(ref[0]).numpy(), rate=SAMPLE_RATE))
    print("  OLA FIR:")
    display(Audio(norm(ola[0]).numpy(), rate=SAMPLE_RATE))
    return ref, ola

## 1. Vowel sounds

Different tongue positions produce different vowels.

In [ ]:
vowels = {
    "/ɑ/ (father)": dict(tongueIndex=12.9, tongueDiameter=2.43),
    "/i/ (feet)": dict(tongueIndex=27.2, tongueDiameter=2.05),
    "/u/ (boot)": dict(tongueIndex=22.8, tongueDiameter=2.05),
    "/ɛ/ (bed)": dict(tongueIndex=20.0, tongueDiameter=2.3),
    "/ʌ/ (but)": dict(tongueIndex=17.0, tongueDiameter=2.3),
}

for label, kw in vowels.items():
    params = make_params(duration_s=1.5, frequency=350, tenseness=0.6, **kw)
    play_both(params, label=label)

## 2. Waveform comparison

Visual comparison of reference vs OLA FIR on a single vowel.

In [ ]:
params = make_params(duration_s=1.0, frequency=200, tenseness=0.6,
                     tongueIndex=27.2, tongueDiameter=2.05,
                     noise=0.0, vibratoGain=0.0, vibratoWobble=0.0)
ref, ola = play_both(params, seed=0, label="/i/ waveform comparison")

# Show a short window
start, end = 2 * SAMPLES_PER_FRAME, 2 * SAMPLES_PER_FRAME + 800
t = torch.arange(end - start) / SAMPLE_RATE * 1000  # ms

fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
axes[0].plot(t, ref[0, start:end].numpy(), label="reference", linewidth=0.8)
axes[0].set_ylabel("reference")
axes[1].plot(t, ola[0, start:end].numpy(), color="C1", label="OLA FIR", linewidth=0.8)
axes[1].set_ylabel("OLA FIR")
err = (ola[0, start:end] - ref[0, start:end]).numpy()
axes[2].plot(t, err, color="C3", linewidth=0.8)
axes[2].set_ylabel("error")
axes[2].set_xlabel("time (ms)")
fig.suptitle("/i/ — reference vs OLA FIR (static params, 18 ms window)")
plt.tight_layout()
plt.show()

print(f"peak |error|: {abs(err).max():.4f}  |  signal peak: {ref[0, start:end].abs().max():.4f}")

## 3. Spectrogram comparison

In [ ]:
import numpy as np

params = make_params(duration_s=1.0, frequency=200, tenseness=0.6,
                     tongueIndex=27.2, tongueDiameter=2.05,
                     noise=0.0, vibratoGain=0.0, vibratoWobble=0.0)

with torch.no_grad():
    ref = pink_trombone(params, seed=0)[0].numpy()
    ola = pink_trombone_ola(params, seed=0, ir_length=4096)[0].numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, sig, title in [(axes[0], ref, "Reference"), (axes[1], ola, "OLA FIR")]:
    ax.specgram(sig, Fs=SAMPLE_RATE, NFFT=1024, noverlap=512, cmap="inferno")
    ax.set_ylim(0, 5000)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("frequency (Hz)")
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 4. Vowel glide (time-varying params)

Sweep tongue index smoothly /i/ → /ɑ/.  The OLA approximation uses per-frame
filters sampled at frame midpoints, so slowly-varying params should still be close.

In [ ]:
T = 38  # ~3 seconds
params = make_params(
    duration_s=T / CONTROL_RATE,
    tongueIndex=torch.linspace(27.2, 12.9, T),
    tongueDiameter=torch.linspace(2.05, 2.43, T),
    frequency=350,
    tenseness=0.6,
)
play_both(params, label="Glide /i/ → /ɑ/")

## 5. Constriction sweep

Open → closed: from vowel to stop consonant.

In [ ]:
T = 38
params = make_params(
    duration_s=T / CONTROL_RATE,
    frequency=140,
    tenseness=0.6,
    tongueIndex=12.9,
    tongueDiameter=2.43,
    constrictionIndex=30.0,
    constrictionDiameter=torch.linspace(2.0, -0.5, T),
)
play_both(params, label="Constriction sweep: open → closed")

## 6. Fricative

Turbulence path: narrow constriction mid-tract.

In [ ]:
params = make_params(
    duration_s=0.5,
    frequency=140,
    tenseness=0.1,
    constrictionIndex=32.0,
    constrictionDiameter=0.5,
)
play_both(params, label="Fricative mid-tract (c_idx=32, c_diam=0.5)")

## 7. Speed benchmark

Compare wall-clock time across clip lengths.

In [ ]:
import gc

durations = [1, 2, 4, 8]
results = []

for dur in durations:
    params = make_params(duration_s=dur, frequency=350, tenseness=0.6,
                         tongueIndex=27.2, tongueDiameter=2.05,
                         noise=0.0, vibratoGain=0.0, vibratoWobble=0.0)
    times = {}
    for name, fn in [("reference", pink_trombone), ("OLA FIR", pink_trombone_ola)]:
        # warm-up
        with torch.no_grad():
            fn(params, seed=0)
        gc.collect()
        t0 = time.perf_counter()
        with torch.no_grad():
            for _ in range(3):
                fn(params, seed=0)
        times[name] = (time.perf_counter() - t0) / 3

    results.append((dur, times))
    print(f"{dur}s clip: reference={times['reference']*1000:.0f} ms, "
          f"OLA FIR={times['OLA FIR']*1000:.0f} ms, "
          f"speedup={times['reference']/times['OLA FIR']:.1f}x")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(durations, [r[1]["reference"]*1000 for r in results], "o-", label="reference")
ax.plot(durations, [r[1]["OLA FIR"]*1000 for r in results], "s-", label="OLA FIR")
ax.set_xlabel("clip duration (s)")
ax.set_ylabel("wall time (ms)")
ax.set_title("Synthesis time vs clip length (CPU)")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Accuracy vs IR length

Shorter IRs are faster but less accurate.  The 0.999-per-sample damping means the
impulse response decays to ~1.7% of its initial amplitude at 4096 taps (~93 ms).

In [ ]:
params = make_params(duration_s=0.4, frequency=200, tenseness=0.6,
                     tongueIndex=27.2, tongueDiameter=2.05,
                     noise=0.0, vibratoGain=0.0, vibratoWobble=0.0)

with torch.no_grad():
    ref = pink_trombone(params, seed=0)

ir_lengths = [256, 512, 1024, 2048, 4096]
errors = []

skip = SAMPLES_PER_FRAME
ref_rms = ref[:, skip:].pow(2).mean().sqrt().item()

for L in ir_lengths:
    with torch.no_grad():
        ola = pink_trombone_ola(params, seed=0, ir_length=L)
    err_rms = (ola[:, skip:] - ref[:, skip:]).pow(2).mean().sqrt().item()
    rel = err_rms / ref_rms
    errors.append(rel)
    print(f"L={L:5d}: relative error = {rel:.4f} ({rel*100:.2f}%)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(ir_lengths, [e * 100 for e in errors], "o-")
ax.set_xlabel("IR length (taps)")
ax.set_ylabel("relative RMS error (%)")
ax.set_title("OLA FIR accuracy vs IR length")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Gradient comparison

Gradients of silence-loss w.r.t. each parameter, for both methods.

In [ ]:
params_ref = make_params(duration_s=2.0)
params_ola = params_ref.clone()
params_ref.requires_grad_(True)
params_ola.requires_grad_(True)

loss_ref = pink_trombone(params_ref, seed=0).pow(2).mean()
loss_ref.backward()

loss_ola = pink_trombone_ola(params_ola, seed=0).pow(2).mean()
loss_ola.backward()

grad_ref = params_ref.grad[0].abs().mean(dim=0).detach()
grad_ola = params_ola.grad[0].abs().mean(dim=0).detach()

x = range(N_PARAMS)
fig, ax = plt.subplots(figsize=(10, 4))
width = 0.35
ax.bar([i - width/2 for i in x], grad_ref.numpy(), width, label="reference")
ax.bar([i + width/2 for i in x], grad_ola.numpy(), width, label="OLA FIR", alpha=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(PARAM_NAMES, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("|∂loss/∂param|")
ax.set_title("Parameter gradients (silence loss, 2 s clip)")
ax.legend()
plt.tight_layout()
plt.show()